In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import GradientBoostingClassifier

import matplotlib.pyplot as plt
from sklearn.model_selection import RandomizedSearchCV # 튜닝

In [6]:
df = pd.read_csv('./src/ai4i2020.csv')
df

,UDI,Product ID,Type,Air temperature [K],Process temperature [K],Rotational speed [rpm],Torque [Nm],Tool wear [min],Machine failure,TWF,HDF,PWF,OSF,RNF
0,1,M14860,M,298.1,308.6,1551,42.8,0,0,0,0,0,0,0
1,2,L47181,L,298.2,308.7,1408,46.3,3,0,0,0,0,0,0
2,3,L47182,L,298.1,308.5,1498,49.4,5,0,0,0,0,0,0
3,4,L47183,L,298.2,308.6,1433,39.5,7,0,0,0,0,0,0
4,5,L47184,L,298.2,308.7,1408,40.0,9,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,M24855,M,298.8,308.4,1604,29.5,14,0,0,0,0,0,0
9996,9997,H39410,H,298.9,308.4,1632,31.8,17,0,0,0,0,0,0
9997,9998,M24857,M,299.0,308.6,1645,33.4,22,0,0,0,0,0,0
9998,9999,H39412,H,299.0,308.7,1408,48.5,25,0,0,0,0,0,0


In [ ]:
# Product ID 제거
df = df.drop(columns=["Product ID"])

# Type One-Hot Encoding
df = pd.get_dummies(df, columns=["Type"], drop_first=True)

###################################################
# Feature Engineering
###################################################

# 출력
df["Power"] = df["Rotational speed [rpm]"] * df["Torque [Nm]"]

# 온도차
df["Temp_diff"] = (
    df["Process temperature [K]"] -
    df["Air temperature [K]"]
)

# 출력 × 공구마모
df["Power_wear"] = (
    df["Power"] * df["Tool wear [min]"]
)

# 토크 / 회전속도
df["Torque_per_RPM"] = (
    df["Torque [Nm]"] /
    (df["Rotational speed [rpm]"] + 1)
)

# 공구 마모 * 토크
df["Torque_wear"] = (
    df["Torque [Nm]"] *
    df["Tool wear [min]"]
)
# 온도차 * 토크
df["Temp_Torque"] = (
    df["Temp_diff"] *
    df["Torque [Nm]"]
)
# 공정온도 / 회기온도
df["Temp_ratio"] = (
    df["Process temperature [K]"] /
    df["Air temperature [K]"]
)

###################################################
# X, y 분리
###################################################

X = df.drop(columns=[
    "Machine failure",
    "TWF",
    "HDF",
    "PWF",
    "OSF",
    "RNF"
])

y = df["Machine failure"]


param_dist = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [5, 10, 15, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"],
    "class_weight": [
        "balanced",
        {0:1, 1:3},
        {0:1, 1:5},
        {0:1, 1:8}
    ]
}



ValueError: Cannot specify both 'axis' and 'index'/'columns'

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=49,
    stratify=y
)

In [ ]:
scaler = MinMaxScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)

X_train_sm, y_train_sm = sm.fit_resample(
    X_train,
    y_train
)

In [ ]:
rf = RandomForestClassifier(random_state=42)

random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=30,
    cv=5,
    scoring="recall",
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train_sm, y_train_sm)

print("Best Parameters")
print(random_search.best_params_)

print()

print("Best Recall")
print(random_search.best_score_)





In [ ]:
best_rf = random_search.best_estimator_

pred = best_rf.predict(X_test)

In [ ]:
print(classification_report(y_test, pred))

ConfusionMatrixDisplay.from_predictions(y_test, pred)

plt.show()